# 3 — Sentiment / Intent Classifier (OpenRouter LLM)

> **Yazılı karşılığı:** [`docs/3_sentiment_classifier.docx`](../docs/3_sentiment_classifier.docx) — metodoloji, prompt mimarisi, hata analizi ve Phase 4 entegrasyonu docx'te. Bu defter `artifacts/sentiment_predictions/glm-4-5-air_test.parquet` ve `artifacts/sentiment_metrics.json` üzerinden canlı sayıları + tabloları yeniden üretir; OpenRouter'a çağrı yapmaz.

## 1. Kurulum ve veri yükleme

`scripts/evaluate_openrouter_sentiment.py` test seti boyunca model çağrılarını checkpoint-resume mantığıyla `artifacts/sentiment_predictions/{model}_test.parquet` altına yazar. `scripts/build_3_sentiment_docx.py` aynı parquet'i canonical metriklere dönüştürür ve docx'i kurar. Bu defter o iki dosyayı sade ekran tabloları + grafiklerle bir araya getirir.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from lead_priority.features.constants import SEED
from lead_priority.models import SENTIMENT_CLASSES, SENTIMENT_SCORE_MAP
from lead_priority.settings import REPO_ROOT

INTERACTIONS = REPO_ROOT / 'data' / 'synthetic' / 'interactions.parquet'
PREDICTIONS = REPO_ROOT / 'artifacts' / 'sentiment_predictions' / 'glm-4-5-air_test.parquet'
METRICS_PATH = REPO_ROOT / 'artifacts' / 'sentiment_metrics.json'
FIGURES_DIR = REPO_ROOT / 'artifacts' / 'figures'

interactions = pd.read_parquet(INTERACTIONS)
predictions = pd.read_parquet(PREDICTIONS)
metrics = json.loads(METRICS_PATH.read_text())
print(f'interactions={len(interactions)}, predictions={len(predictions)}, model={metrics["model_name"]}')

## 2. Test bölmesi ve sınıf × dil dağılımı

Eval CLI ile aynı seed (42) ve aynı iki ardışık stratified split — sonuç: 9.240 interaktif notun %10'u (924) test seti. Stratifikasyon anahtarı `attitude + language` (12 strat), test seti hem sınıf hem dil dağılımını korur.

In [ ]:
from sklearn.model_selection import train_test_split

strata = interactions['attitude'].astype(str) + '_' + interactions['language'].astype(str)
_, test_part = train_test_split(
    interactions, test_size=0.10, random_state=SEED, stratify=strata, shuffle=True,
)
test_part = test_part.reset_index(drop=True)
merged = test_part.merge(predictions, on='lead_id', how='inner', suffixes=('', '_pred'))
merged = merged.drop(columns=[c for c in ['language_pred'] if c in merged.columns])
if 'true_attitude' not in merged.columns:
    merged['true_attitude'] = merged['attitude']
print(f'test={len(test_part)}, merged={len(merged)} (eşleşmeyen lead_id: {len(test_part) - len(merged)})')
pd.crosstab(merged['true_attitude'], merged['language'], margins=True).reindex(
    index=list(SENTIMENT_CLASSES) + ['All'], columns=['tr', 'en', 'mix', 'All']
)

## 3. Genel performans

Üç temel metrik: **accuracy** (sade ama dengesizliğe duyarsız), **F1-macro** (sınıf dengesizliğini düzeltir — az örneklenmiş sınıfa eşit ağırlık), **F1-weighted** (örnek sayısına göre ağırlıklı — production'da gerçek kullanıcı deneyimini en iyi yansıtan tek skor). Bu üçü de canonical `sentiment_metrics.json`'dan okunur.

In [ ]:
overall = metrics['overall']
ci = (overall['f1_macro_ci_lower'], overall['f1_macro_ci_upper'])
print(f"Accuracy     : {overall['accuracy']:.4f}")
print(f"F1-macro     : {overall['f1_macro']:.4f}  (95% CI [{ci[0]:.4f}, {ci[1]:.4f}])")
print(f"F1-weighted  : {overall['f1_weighted']:.4f}")
print(f"Test set n   : {metrics['n_test']}")
print(f"Model        : {metrics['model_name']}")

## 4. Sınıf-başına precision / recall / F1

Sınıf desteği (support) test setindeki gerçek örnek sayısını verir. `positive_engagement` baskın (~%34); `objection` en az örneklenmiş (~%16).

In [ ]:
per_class = pd.DataFrame(metrics['per_class']).set_index('class')
per_class[['precision', 'recall', 'f1', 'support']].round(4)

## 5. Confusion matrix

Diyagonal hücreler doğru tahminler. Off-diagonal hata türlerini gösterir — özellikle `neutral` sınıfı diğer üç sınıfa sık sızıyor; model bu sınıfı 'belirsiz' bir kovaya doğru aktarıyor (örn. `neutral → positive_engagement` en sık hata).

In [ ]:
from IPython.display import Image
Image(filename=str(FIGURES_DIR / '3_confusion_matrix.png'))

In [ ]:
cm = confusion_matrix(
    merged['true_attitude'], merged['predicted_attitude'], labels=list(SENTIMENT_CLASSES)
)
pd.DataFrame(cm, index=list(SENTIMENT_CLASSES), columns=list(SENTIMENT_CLASSES))

## 6. Per-language F1 — multilingual baseline kanıtı

Veri TR / EN / Mix karışık geliyor. Multilingual model iddiası için her dil diliminde F1'in tutarlı olması gerekir. Mix bandında küçük bir gerileme (~3-5 pp) code-switching üzerindeki baseline limitidir.

In [ ]:
per_lang = pd.DataFrame(metrics['per_language']).T
per_lang[['n', 'accuracy', 'f1_macro', 'f1_weighted']].round(4)

In [ ]:
Image(filename=str(FIGURES_DIR / '3_per_language_f1.png'))

## 7. Bootstrap CI — F1-macro

Tek seed-tek metrik raporu yeterli değil; test seti tek bir örneklemden ibarettir. **1.000 stratified bootstrap iterasyonu (seed=42)** F1-macro'nun etrafına 95% güven aralığı yerleştirir. Dar bir CI test setinin yeterince büyük olduğunu ve nokta tahmininin güvenilir olduğunu söyler.

In [ ]:
Image(filename=str(FIGURES_DIR / '3_bootstrap_ci.png'))

## 8. Hata analizi

Yanlış sınıflandırma sayısı + en sık 10 hata türü + dile göre dağılım. `neutral → positive_engagement` ve `disengaged → neutral` en sık ikilik; her ikisi de modelin 'belirsiz' notları daha pozitif kovaya kaydırma eğilimini gösterir.

In [ ]:
errors = merged[merged['predicted_attitude'] != merged['true_attitude']]
print(f'Toplam hata: {len(errors)} / {len(merged)} ({len(errors)/len(merged):.2%})')
print()
print('Dile göre:')
print(errors['language'].value_counts())
print()
print('Gerçek sınıfa göre:')
print(errors['true_attitude'].value_counts())

In [ ]:
Image(filename=str(FIGURES_DIR / '3_error_breakdown.png'))

### 8.1 Örnek yanlış sınıflandırmalar

Aşağıdaki tablo `sentiment_metrics.json` içinden 12 yanlış sınıflandırılan örneği gösterir — dil + sınıf kombinasyonu + kısaltılmış metin.

In [ ]:
samples = pd.DataFrame(metrics['sample_errors'])
samples

## 9. Phase 4 entegrasyonu — sentiment → priority

Sentiment tek başına önceliklendirme için yeterli değil; Phase 4 weighted average sentiment'i lead scoring olasılığı ile birleştirir:

```
priority = 0.6 · P(conversion) + 0.4 · sentiment_score(attitude)
```

Sınıf → ağırlık haritası `notes/next_steps.md` §4'teki crosstab analizine dayanır (objection sınıfının %53'ü dönüşüyor → 0,65 ağırlık).

In [ ]:
pd.Series(SENTIMENT_SCORE_MAP, name='sentiment_score').rename_axis('attitude').to_frame()

In [ ]:
merged['sentiment_score'] = merged['predicted_attitude'].map(SENTIMENT_SCORE_MAP)
merged.groupby('predicted_attitude', observed=True)['sentiment_score'].describe()[['count', 'mean']]

## 10. Sonraki adımlar

Detaylı next-step listesi (encoder fine-tune, prompt iteration, active learning, multilingual fairness audit) `docs/3_sentiment_classifier.docx` §11'de. Phase 4 ile entegrasyon `src/lead_priority/models/sentiment.py` içindeki `OpenRouterSentiment.predict_score()` üzerinden — Phase 4 birleşik skor hesabı bu fonksiyonu çağırır.